### We want to calculate the smallest $ \eta $ for our experiments to see if it is smaller than our grid cell size $ \Delta x, \Delta z $

In [1]:
#import packages

using Oceananigans
using CairoMakie
using NCDatasets
using Statistics
using Printf


In [ ]:
#import datasets we need the oceanostics file for the dissipation rate
# we will analyze the 1e5, 1e6, and 1e7 Ra cases with hill

ds_1 = NCDataset(string("/Users/hfdrake/code/HorizontalConvection/output/turbulent_onehill_0.6_Ra100000.0_coldstart_oceanostics.nc"));
ds_2 = NCDataset(string("/Users/hfdrake/code/HorizontalConvection/output/turbulent_onehill_0.6_Ra1.0e6_coldstart_oceanostics.nc"));
ds_3 = NCDataset(string("/Users/hfdrake/code/HorizontalConvection/output/turbulent_onehill_0.6_Ra5.0e6_coldstart_oceanostics.nc"));
ds_4 = NCDataset(string("/Users/hfdrake/code/HorizontalConvection/output/turbulent_onehill_0.6_Ra1.0e7_coldstart_oceanostics.nc"));

list_hill_datasets = [ds_1, ds_2, ds_3, ds_4];


In [35]:
#define the grid cell sizes Δx is always gonna be larger than Δz so we only need to check Δz

Δz_1 = 1/169
Δz_2 = 1/301
Δz_3 = 1/450
Δz_4 = 1/535;

Δzs = [Δz_1, Δz_2, Δz_3, Δz_4];

In [37]:
ε_1 = ds_1["ε"][4+1:end-4, 1, 4+1:end-4, :]
ε_2 = ds_2["ε"][4+1:end-4, 1, 4+1:end-4, :]
ε_3 = ds_3["ε"][4+1:end-4, 1, 4+1:end-4, :]
ε_4 = ds_4["ε"][4+1:end-4, 1, 4+1:end-4, :];

epsilons = [ε_1, ε_2, ε_3, ε_4];

In [40]:
function get_η(ε)
    η = ε.^(-1/4)
    return η
end

get_η (generic function with 1 method)

In [45]:
η_1 = get_η(ε_1)
η_min_1 = minimum(x for x in η_1 if x > 0)
if Δz_1 < η_min_1
    @printf("The Kolmogorov scale is resolved for Ra = 1e5 with Δz = %.5f and η_min = %.5f\n", Δz_1, η_min_1)
else
    @printf("The Kolmogorov scale is NOT resolved for Ra = 1e5 with Δz = %.5f and η_min = %.5f\n", Δz_1, η_min_1)
end

The Kolmogorov scale is resolved for Ra = 1e5 with Δz = 0.00592 and η_min = 1.72137


In [46]:
function is_kolmogorov_resolved(Δz, ε)
    η = get_η(ε)
    η_min = minimum(x for x in η if x > 0)
    if Δz < η_min
        return true, η_min
    else
        return false, η_min
    end
end

is_kolmogorov_resolved (generic function with 1 method)

In [51]:
for (i,j) in zip(epsilons, Δzs)
    resolved, η_min = is_kolmogorov_resolved(j, i)
    if resolved
        @printf("The Kolmogorov scale is resolved with Δz = %.5f and η_min = %.5f\n", j, η_min)
    else
        @printf("The Kolmogorov scale is NOT resolved with Δz = %.5f and η_min = %.5f\n", j, η_min)
    end
end

The Kolmogorov scale is resolved with Δz = 0.00592 and η_min = 1.72137
The Kolmogorov scale is resolved with Δz = 0.00332 and η_min = 1.52066
The Kolmogorov scale is resolved with Δz = 0.00222 and η_min = 1.37774
The Kolmogorov scale is resolved with Δz = 0.00187 and η_min = 1.17849


### Now calculate the ratio $ r(x,t) = \frac{\Delta z}{\eta(x,t)} $ 

from this we can see if the dissipative scales are resolved:

if $ r << 1 $ the dissipative scales are well resolved

if $ r \geq 1 $ the smallest dissipative structures are under-resolved

In [54]:
filter(x -> isfinite(x) && x > 0, vec(η_1))


107229-element Vector{Float64}:
 15.823489696538159
 15.83324494110614
 15.852862507197809
 15.88231878084405
 15.921671912735365
 15.970992251504148
 16.03036720931224
 16.099906628992873
 16.17974750243634
 16.270058390978058
  ⋮
 10.397709325080095
 10.430320668310832
 10.459752263943644
 10.485695881283865
 10.50780323621001
 10.525693044980127
 10.539161096309389
 10.549035630365054
 10.556461677380364

In [57]:
function get_ratio(Δz, ε)
    η = get_η(ε)
    η_data = filter(x -> isfinite(x) && x > 0, vec(η))
    r = Δz ./ η_data
    return r
end

function compute_statistics(Δz, ε)
    r = get_ratio(Δz, ε)
    
    r_min = minimum(r)
    r_max = maximum(r)
    r_median = median(r)
    r_mean = mean(r)
    r_95 = quantile(r, 0.95)

    frac_r_greater_than_1 = count(>(1.0), r) / length(r)

    return Dict(
        "r_min" => r_min,
        "r_max" => r_max,
        "r_median" => r_median,
        "r_mean" => r_mean,
        "r_95" => r_95,
        "frac_r_greater_than_1" => frac_r_greater_than_1
    )
end

compute_statistics (generic function with 2 methods)

In [ ]:
#statistics for Ra = 1e5
compute_statistics(Δz_1, ε_1)

Dict{String, Float64} with 6 entries:
  "r_max"                 => 0.00343748
  "r_mean"                => 0.00105071
  "r_95"                  => 0.00178653
  "r_median"              => 0.00102387
  "frac_r_greater_than_1" => 0.0
  "r_min"                 => 4.87161e-5

In [ ]:
#statistics for Ra = 1e6
compute_statistics(Δz_2, ε_2)

Dict{String, Float64} with 6 entries:
  "r_max"                 => 0.00218474
  "r_mean"                => 0.000372368
  "r_95"                  => 0.000779692
  "r_median"              => 0.000334817
  "frac_r_greater_than_1" => 0.0
  "r_min"                 => 9.53797e-6

In [ ]:
#statistics for Ra = 5e6
compute_statistics(Δz_3, ε_3)

Dict{String, Float64} with 6 entries:
  "r_max"                 => 0.00161294
  "r_mean"                => 0.000195202
  "r_95"                  => 0.000419271
  "r_median"              => 0.000169148
  "frac_r_greater_than_1" => 0.0
  "r_min"                 => 5.5798e-7

In [ ]:
#statistics for Ra = 1e7
compute_statistics(Δz_4, ε_4)

Dict{String, Float64} with 6 entries:
  "r_max"                 => 0.00158606
  "r_mean"                => 0.000147998
  "r_95"                  => 0.00032198
  "r_median"              => 0.000122828
  "frac_r_greater_than_1" => 0.0
  "r_min"                 => 3.91364e-7

SO, the results of the ratio analysis show that, since r is never greater than one that the grid size is always smaller than the kolmogorov scale, which means that the kolmogorov scale is resolved. 